### 4. Exercises (English Version)

Using the `sqlite3` library:

* Create a new database called **`district_court.db`**, and inside it create two tables:  
  - the first one should be called **`judges`** and contain the columns: `id`, `first_name`, `last_name`, `division`, `years_experience`;  
  - the second one should be called **`open_cases`** and contain the columns: `case_number`, `division`, `judge` (this last one is a **FOREIGN KEY** referring to `id` in the first table);

* Fill both tables, **`judges`** and **`open_cases`**, with the following records, respectively:  
  - **judges:**  
    (Kazimierz, Sprawiedliwy, I Civil, 15),  
    (Antoni, Rozjemca, I Civil, 12),  
    (Remigiusz, Surowy, I Criminal, 20),  
    (Janina, Sroga, I Criminal, 20);  
  - **open_cases:**  
    (I C 187/24, I Civil, 1),  
    (I C 116/24, I Civil, 2),  
    (I K 17/24, I Criminal, 3),  
    (I K 29/24, I Criminal, 4);  
  Use **parameterized queries** when inserting the records.

* Join the two tables using **INNER JOIN**; the `judge` column from the second table refers to the judge’s `id` from the first table. Display selected columns from the resulting joined table.


In [2]:
import sqlite3

# Create a NEW temporary database in RAM (fast, clean, no file locking issues)
conn = sqlite3.connect(":memory:")

# Enable foreign key constraints (disabled by default in SQLite)
conn.execute("PRAGMA foreign_keys = ON;")

# Create a cursor object to execute SQL commands
cursor = conn.cursor()

# ---------------------------
# CREATE TABLES
# ---------------------------

# Table 1: judges
cursor.execute("""
CREATE TABLE judges (
    id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    division TEXT NOT NULL,
    years_experience INTEGER NOT NULL
);
""")

# Table 2: open_cases with a FOREIGN KEY referencing judges(id)
cursor.execute("""
CREATE TABLE open_cases (
    case_number TEXT PRIMARY KEY,
    division TEXT NOT NULL,
    judge INTEGER NOT NULL,
    FOREIGN KEY (judge) REFERENCES judges(id)
);
""")

# ---------------------------
# INSERT DATA (parameterized)
# ---------------------------

judges_data = [
    ("Kazimierz", "Sprawiedliwy", "I Civil", 15),
    ("Antoni", "Rozjemca", "I Civil", 12),
    ("Remigiusz", "Surowy", "I Criminal", 20),
    ("Janina", "Sroga", "I Criminal", 20),
]

open_cases_data = [
    ("I C 187/24", "I Civil", 1),
    ("I C 116/24", "I Civil", 2),
    ("I K 17/24", "I Criminal", 3),
    ("I K 29/24", "I Criminal", 4),
]

# Insert judges
cursor.executemany("""
INSERT INTO judges (first_name, last_name, division, years_experience)
VALUES (?, ?, ?, ?);
""", judges_data)

# Insert open cases
cursor.executemany("""
INSERT INTO open_cases (case_number, division, judge)
VALUES (?, ?, ?);
""", open_cases_data)

# Save changes to the database
conn.commit()

# ---------------------------
# INNER JOIN QUERY
# ---------------------------

cursor.execute("""
SELECT 
    oc.case_number, 
    oc.division, 
    j.first_name, 
    j.last_name
FROM open_cases oc
JOIN judges j ON oc.judge = j.id
ORDER BY oc.case_number;
""")

# Print query results
for row in cursor.fetchall():
    print(row)

# Close connection (RAM database disappears automatically)
conn.close()


('I C 116/24', 'I Civil', 'Antoni', 'Rozjemca')
('I C 187/24', 'I Civil', 'Kazimierz', 'Sprawiedliwy')
('I K 17/24', 'I Criminal', 'Remigiusz', 'Surowy')
('I K 29/24', 'I Criminal', 'Janina', 'Sroga')
